# Rankings and Report

Score every product in the catalog for one representative user under a fixed context, then show the top 5 products by predicted click probability.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

data_dir = Path("../data")
processed_dir = data_dir / "processed"

train_df = pd.read_csv(processed_dir / "train_feature_engineered.csv")
validation_df = pd.read_csv(processed_dir / "validation_feature_engineered.csv")
train_df = train_df.sample(n=1000, random_state=42)

target = "clicked"

X_train = pd.get_dummies(train_df.drop(columns=[target]), dummy_na=True)
y_train = train_df[target]
X_validation = pd.get_dummies(validation_df.drop(columns=[target]), dummy_na=True)
y_validation = validation_df[target]

X_train.columns = X_train.columns.astype(str).str.replace(r"[\[\]<>]", "", regex=True)
X_validation.columns = X_validation.columns.astype(str).str.replace(r"[\[\]<>]", "", regex=True)
X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)

X_train.shape, X_validation.shape

((1000, 48), (67500, 48))

In [2]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count if positive_count else 1.0

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        random_state=42,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    ),
}

results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    train_proba = model.predict_proba(X_train)[:, 1]
    validation_proba = model.predict_proba(X_validation)[:, 1]
    train_auc = roc_auc_score(y_train, train_proba)
    validation_auc = roc_auc_score(y_validation, validation_proba)
    results.append(
        {
            "model": model_name,
            "train_auc": train_auc,
            "validation_auc": validation_auc,
            "gap": train_auc - validation_auc,
        }
    )

results_df = pd.DataFrame(results).sort_values(
    ["validation_auc", "gap"], ascending=[False, True]
).reset_index(drop=True)
best_row = results_df.iloc[0]
best_model_name = best_row["model"]

print("Best model by validation AUC-ROC, then smallest train-validation gap:")
print(best_row[["model", "train_auc", "validation_auc", "gap"]])
results_df

Best model by validation AUC-ROC, then smallest train-validation gap:
model             Gradient Boosting
train_auc                  0.999716
validation_auc             0.683845
gap                        0.315871
Name: 0, dtype: object


,model,train_auc,validation_auc,gap
0,Gradient Boosting,0.999716,0.683845,0.315871
1,Random Forest,1.000000,0.672083,0.327917
2,XGBoost,1.000000,0.657004,0.342996


In [3]:
joined = pd.read_csv(data_dir / "interim" / "joined_dataset.csv")
catalog = pd.read_csv(data_dir / "raw" / "product_catalog.csv")

representative_user_id = joined["user_id"].value_counts().idxmax()
user_history = joined.loc[joined["user_id"] == representative_user_id]

user_profile = {
    "age": int(user_history["age"].median()),
    "gender": user_history["gender"].mode().iloc[0],
    "city": user_history["city"].mode().iloc[0],
    "device_os": user_history["device_os"].mode().iloc[0],
}

city_freq_map = joined["city"].fillna("missing").astype(str).value_counts(normalize=True).to_dict()
product_name_freq_map = joined["product_name"].fillna("missing").astype(str).value_counts(normalize=True).to_dict()
brand_freq_map = joined["brand"].fillna("missing").astype(str).value_counts(normalize=True).to_dict()

def build_catalog_features(catalog_df: pd.DataFrame, user: dict) -> pd.DataFrame:
    out = catalog_df.copy()
    out["slot_position"] = 1
    out["hour_of_day"] = 19
    out["age"] = user["age"]
    out["city"] = user["city"]
    out["gender"] = user["gender"]
    out["platform"] = "mobile_app"
    out["device_os"] = user["device_os"]
    out["page_type"] = "home"
    out["day_of_week"] = "Tue"
    out["city_freq"] = out["city"].fillna("missing").astype(str).map(city_freq_map).fillna(0.0)
    out["product_name_freq"] = out["product_name"].fillna("missing").astype(str).map(product_name_freq_map).fillna(0.0)
    out["brand_freq"] = out["brand"].fillna("missing").astype(str).map(brand_freq_map).fillna(0.0)
    age_edges = [-float("inf"), 26, 32, 38, 44, float("inf")]
    out["age_bucket"] = pd.cut(out["age"], bins=age_edges, include_lowest=True).astype(str)
    out["log_price_usd"] = np.log1p(out["price_usd"])
    out["discounted_price"] = out["price_usd"] * (1 - out["discount_pct"] / 100)
    out["price_per_review"] = out["price_usd"] / (out["review_count"].fillna(0) + 1)
    out["slot_position_zscore"] = 0.0
    out["frequency_density"] = out[["city_freq", "product_name_freq", "brand_freq"]].mean(axis=1)
    return out

catalog_features = build_catalog_features(catalog, user_profile)
catalog_X = pd.get_dummies(catalog_features, dummy_na=True)
catalog_X.columns = catalog_X.columns.astype(str).str.replace(r"[\[\]<>]", "", regex=True)
catalog_X = catalog_X.reindex(columns=X_train.columns, fill_value=0)

catalog_X.shape

(1200, 48)

In [4]:
if best_model_name == "Random Forest":
    final_model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )
elif best_model_name == "Gradient Boosting":
    final_model = GradientBoostingClassifier()
else:
    final_model = XGBClassifier(
        random_state=42,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    )

final_model.fit(X_train, y_train)
catalog_scores = catalog.copy()
catalog_scores["predicted_click_probability"] = final_model.predict_proba(catalog_X)[:, 1]

top_5 = catalog_scores.sort_values("predicted_click_probability", ascending=False).head(5)[
    ["product_name", "brand", "category", "price_usd", "predicted_click_probability"]
]

print(f"Representative user_id: {representative_user_id}")
print("Fixed context: platform=mobile_app, page_type=home, day_of_week=Tue, hour_of_day=19, slot_position=1")
top_5

Representative user_id: 7077
Fixed context: platform=mobile_app, page_type=home, day_of_week=Tue, hour_of_day=19, slot_position=1


,product_name,brand,category,price_usd,predicted_click_probability
787,Kellogg's Original Snack Variety Pack,Kellogg's,Grocery,8.49,0.725260
417,Kraft Heinz Value Pack Tea Collection,Kraft Heinz,Grocery,10.99,0.722342
622,The Hidden Harbor - Paperback,Macmillan,Books,10.99,0.672464
156,Kraft Heinz Original Organic Soup Pack,Kraft Heinz,Grocery,10.99,0.671733
930,Kellogg's Family Size Coffee Blend,Kellogg's,Grocery,9.95,0.668674


## Summary

- Selected a representative user from the most frequent `user_id` in the joined dataset.
- Scored the entire product catalog in a fixed context: mobile app, homepage, Tuesday 7 PM, slot 1.
- Reported the top 5 products by predicted click probability.